# 02 — Modèles de base (baseline)

Un Random Forest et un XGBoost par tâche Tox21, évalués par **ROC-AUC** sur le jeu de test (scaffold split).

In [ ]:
# !pip install rdkit shap xgboost pyarrow -q   # Colab
import sys; sys.path.append('src')
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

df = pd.read_parquet('data/tox21_clean.parquet')
X = np.hstack([np.load('data/X_morgan.npy'), pd.read_parquet('data/X_physchem.parquet').values])
TASKS = ['NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase','NR-ER','NR-ER-LBD','NR-PPAR-gamma','SR-ARE','SR-ATAD5','SR-HSE','SR-MMP','SR-p53']

In [ ]:
def run(model_fn, name):
    res = {}
    for t in TASKS:
        lab = df[t].notna().values
        tr = ((df['split'] == 'train').values) & lab
        te = ((df['split'] == 'test').values) & lab
        m = model_fn()
        m.fit(X[tr], df.loc[tr, t])
        res[t] = roc_auc_score(df.loc[te, t], m.predict_proba(X[te])[:, 1])
    return pd.Series(res, name=name)

rf = run(lambda: RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=0), 'RandomForest')
xgb = run(lambda: XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, eval_metric='logloss', n_jobs=-1, random_state=0), 'XGBoost')
results = pd.concat([rf, xgb], axis=1)
results.loc['MOYENNE'] = results.mean()
results.round(3)

In [ ]:
import os, matplotlib.pyplot as plt
os.makedirs('results', exist_ok=True)
ax = results.drop('MOYENNE').plot.bar(figsize=(10, 4))
ax.axhline(0.5, color='gray', ls='--', lw=1)
ax.set_ylabel('ROC-AUC (test, scaffold split)')
ax.set_title('Tox21 : performance par tâche')
plt.tight_layout()
plt.savefig('results/baseline_auc.png', dpi=150)
results.round(3).to_csv('results/baseline_auc.csv')

## Prochaines étapes
- Explicabilité avec SHAP (quelles sous-structures déclenchent l'alerte toxique ?)
- Réseau de neurones sur graphes (GNN)
- Application aux analogues générés dans le dépôt `drug-design-morphine-atropine`